# M5 Feature Engineering — Combined Experiment Runner (NB04 + NB05)

This notebook runs **all internal feature experiments** in one go:

**NB04 experiments (demand history):**
- E0: Baseline
- E1: lag_56
- E2: lag_365
- E3: rstd_28, rmean_56
- E4: rmax_28, rmin_28
- E10: lag_56 + rmax_28 + rmin_28

**NB05 experiments (price / promo / event / hierarchy):**
- E5: price_vs_mean, price_vs_dept_avg
- E6: is_price_drop, is_price_min_7d
- E7: snap_flag, is_event
- E8: dept_avg_demand, store_avg_demand
- E9: E5+E6+E7+E8 combined

**Combination experiments:**
- E7_E1: snap_flag + is_event + lag_56
- E7_E5: snap_flag + is_event + price features
- E9_E1: E9 + lag_56 (best of NB04 + best of NB05)

In [1]:
# =========================
# 1) Colab setup
# =========================
from google.colab import drive
# # drive.mount('/content/drive')  # Colab only  # Colab only


Mounted at /content/drive


In [2]:
# =========================
# 2) Imports
# =========================
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 200)

In [3]:
# =========================
# 3) Configuration
# =========================
# Update paths to match your local environment.
DATA_DIR       = Path("../cleaned_data")  # output of notebook 01
CLEAN_DATA_PATH = DATA_DIR / "long_df_clean.parquet"

OUTPUT_DIR = Path("../cleaned_data/FE_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE    = True
N_SERIES     = 2000
RANDOM_STATE = 1234
VALID_DAYS   = 28

PIPELINE_NAME = "combined_internal_features"

print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)
print("OUTPUT_DIR      :", OUTPUT_DIR)


CLEAN_DATA_PATH: /content/drive/MyDrive/Group Project - Predictive/data/cleaned_data/long_df_clean.parquet
OUTPUT_DIR      : /content/drive/MyDrive/Group Project - Predictive/data/FE_output


In [4]:
# =========================
# 4) Load cleaned dataset
# =========================
needed_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "event_name_1"
]
available_cols = pq.read_schema(CLEAN_DATA_PATH).names
needed_cols = [c for c in needed_cols if c in available_cols]

df = pd.read_parquet(CLEAN_DATA_PATH, columns=needed_cols)
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())

Loaded shape: (16098720, 13)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'date', 'demand', 'sell_price', 'snap_CA', 'snap_TX', 'snap_WI', 'event_name_1']


In [5]:
# =========================
# 5) Sampling
# =========================
df["date"] = pd.to_datetime(df["date"])

if FAST_MODE:
    sampled_ids = (
        df["id"]
        .drop_duplicates()
        .sample(n=min(N_SERIES, df["id"].nunique()), random_state=RANDOM_STATE)
    )
    df = df[df["id"].isin(sampled_ids)].copy()

gc.collect()
df = df.sort_values(["id", "date"]).reset_index(drop=True)

print("Working shape:", df.shape)
print("Unique series:", df["id"].nunique())
print("Date range   :", df["date"].min().date(), "to", df["date"].max().date())

Working shape: (1056000, 13)
Unique series: 2000
Date range   : 2014-12-12 to 2016-05-22


In [6]:
# =========================
# 6) Baseline feature engineering
# =========================
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# Calendar
df["dow"]   = df["date"].dt.dayofweek.astype("int8")
df["month"] = df["date"].dt.month.astype("int8")

# Price
price_grp = df.groupby("id", sort=False)["sell_price"]
df["price_lag_1"] = price_grp.shift(1)
df["price_change"] = (
    (df["sell_price"] - df["price_lag_1"]) / df["price_lag_1"]
).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
df["is_promo"] = (df["sell_price"] < df["price_lag_1"].fillna(df["sell_price"])).astype("int8")

# Demand lags & rolling (shift-first — no leakage)
demand_grp = df.groupby("id", sort=False)["demand"]
df["lag_7"]  = demand_grp.shift(7)
df["lag_28"] = demand_grp.shift(28)
df["rmean_7"]  = df.groupby("id", sort=False)["lag_7"].transform(lambda x: x.rolling(7).mean())
df["rmean_28"] = df.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).mean())

for col in ["sell_price", "price_lag_1", "lag_7", "lag_28", "rmean_7", "rmean_28"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

BASELINE_FEATURE_COLS = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "dow", "month",
    "sell_price", "price_change", "is_promo",
    "lag_7", "lag_28", "rmean_7", "rmean_28"
]

print("Baseline done. df shape:", df.shape)

Baseline done. df shape: (1056000, 22)


In [7]:
# =========================
# 7) Build ALL experimental features
# =========================

# --- NB04 features ---
# E1: lag_56
df["lag_56"] = demand_grp.shift(56)
print("lag_56 done")

# E2: lag_365
df["lag_365"] = demand_grp.shift(365)
print("lag_365 done")

# E3: rstd_28, rmean_56
df["rstd_28"]  = df.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).std())
df["rmean_56"] = df.groupby("id", sort=False)["lag_56"].transform(lambda x: x.rolling(56).mean())
print("E3 done")

# E4: rmax_28, rmin_28
df["rmax_28"] = df.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).max())
df["rmin_28"] = df.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).min())
print("E4 done")

# --- NB05 features ---
# E5: Price Enhancement
df["price_vs_mean"] = df["sell_price"] / df.groupby("id")["sell_price"].transform("mean")
dept_daily_avg = df.groupby(["store_id", "dept_id", "date"])["sell_price"].transform("mean")
df["price_vs_dept_avg"] = df["sell_price"] / dept_daily_avg
df[["price_vs_mean", "price_vs_dept_avg"]] = df[["price_vs_mean", "price_vs_dept_avg"]].astype("float32")
print("E5 done")

# E6: Promotion
df["is_price_drop"] = (df["sell_price"] < df["price_lag_1"]).astype("int8")
df["price_min_7d"]  = df.groupby("id", sort=False)["sell_price"].transform(lambda x: x.shift(1).rolling(7).min())
df["is_price_min_7d"] = (df["sell_price"] == df["price_min_7d"]).astype("int8")
print("E6 done")

# E7: Event & SNAP
df["snap_flag"] = 0
if "snap_CA" in df.columns:
    df.loc[df["state_id"] == "CA", "snap_flag"] = df["snap_CA"]
    df.loc[df["state_id"] == "TX", "snap_flag"] = df["snap_TX"]
    df.loc[df["state_id"] == "WI", "snap_flag"] = df["snap_WI"]
df["snap_flag"] = df["snap_flag"].astype("int8")
if "event_name_1" in df.columns:
    df["event_name_1"] = df["event_name_1"].fillna("NoEvent")
    df["is_event"] = (df["event_name_1"] != "NoEvent").astype("int8")
else:
    df["is_event"] = 0
print("E7 done")

# E8: Hierarchy (shift+rolling — no leakage)
df["dept_avg_demand"] = df.groupby(["store_id", "dept_id", "date"])["demand"].transform("mean")
df["dept_avg_demand"] = df.groupby("id")["dept_avg_demand"].transform(
    lambda x: x.shift(28).rolling(28).mean()
).astype("float32")

df["store_avg_demand"] = df.groupby(["store_id", "date"])["demand"].transform("mean")
df["store_avg_demand"] = df.groupby("id")["store_avg_demand"].transform(
    lambda x: x.shift(28).rolling(28).mean()
).astype("float32")
print("E8 done")

gc.collect()
print("\nAll features built. df shape:", df.shape)

lag_56 done
lag_365 done
E3 done
E4 done
E5 done
E6 done
E7 done
E8 done

All features built. df shape: (1056000, 37)


In [8]:
# =========================
# 8) LightGBM configuration
# =========================
NUM_BOOST_ROUND       = 2000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL          = 100

LGB_PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1
}

print(json.dumps(LGB_PARAMS, indent=2))

{
  "objective": "tweedie",
  "tweedie_variance_power": 1.5,
  "metric": "rmse",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 127,
  "min_data_in_leaf": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "lambda_l1": 0.0,
  "lambda_l2": 0.1,
  "seed": 1234,
  "verbosity": -1
}


In [9]:
# =========================
# 9) Experiment definitions
# =========================

experiment_addons = {
    # --- NB04 (demand history) ---
    "E0":  [],
    "E1":  ["lag_56"],
    "E2":  ["lag_365"],
    "E3":  ["rstd_28", "rmean_56"],
    "E4":  ["rmax_28", "rmin_28"],
    "E10": ["lag_56", "rmax_28", "rmin_28"],

    # --- NB05 (price / promo / event / hierarchy) ---
    "E5":  ["price_vs_mean", "price_vs_dept_avg"],
    "E6":  ["is_price_drop", "is_price_min_7d"],
    "E7":  ["snap_flag", "is_event"],
    "E8":  ["dept_avg_demand", "store_avg_demand"],
    "E9":  ["price_vs_mean", "price_vs_dept_avg",
            "is_price_drop", "is_price_min_7d",
            "snap_flag", "is_event",
            "dept_avg_demand", "store_avg_demand"],

    # --- Combinations ---
    "E7_E1": ["snap_flag", "is_event", "lag_56"],
    "E7_E5": ["snap_flag", "is_event", "price_vs_mean", "price_vs_dept_avg"],
    "E9_E1": ["price_vs_mean", "price_vs_dept_avg",
              "is_price_drop", "is_price_min_7d",
              "snap_flag", "is_event",
              "dept_avg_demand", "store_avg_demand",
              "lag_56"],
}

# Columns that must be non-null for each experiment
experiment_required_addons = {
    "E0":    [],
    "E1":    ["lag_56"],
    "E2":    ["lag_365"],
    "E3":    ["rstd_28", "rmean_56"],
    "E4":    ["rmax_28", "rmin_28"],
    "E10":   ["lag_56", "rmax_28", "rmin_28"],
    "E5":    [],
    "E6":    [],
    "E7":    [],
    "E8":    ["dept_avg_demand", "store_avg_demand"],
    "E9":    ["dept_avg_demand", "store_avg_demand"],
    "E7_E1": ["lag_56"],
    "E7_E5": [],
    "E9_E1": ["dept_avg_demand", "store_avg_demand", "lag_56"],
}

experiments_to_run = [
    "E0",
    "E1", "E2", "E3", "E4", "E10",
    "E5", "E6", "E7", "E8", "E9",
    "E7_E1", "E7_E5", "E9_E1"
]

print(f"Total experiments to run: {len(experiments_to_run)}")
print(experiments_to_run)

Total experiments to run: 14
['E0', 'E1', 'E2', 'E3', 'E4', 'E10', 'E5', 'E6', 'E7', 'E8', 'E9', 'E7_E1', 'E7_E5', 'E9_E1']


In [10]:
# =========================
# 10) Auto-run all experiments
# =========================
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
results = []

for exp_name in experiments_to_run:
    print("=" * 70)
    print(f"Running: {exp_name}")
    added = experiment_addons[exp_name]
    if added:
        print(f"Added features: {added}")
    else:
        print("Baseline only")

    feature_cols          = BASELINE_FEATURE_COLS + added
    required_history_cols = ["lag_28", "rmean_28"] + experiment_required_addons[exp_name]

    model_df = df.dropna(subset=required_history_cols).copy()
    for col in cat_cols:
        model_df[col] = model_df[col].astype("category")

    max_date    = model_df["date"].max()
    valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)
    train_df    = model_df[model_df["date"] < valid_start].copy()
    valid_df    = model_df[model_df["date"] >= valid_start].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["demand"]
    X_valid = valid_df[feature_cols]

    lgb_train = lgb.Dataset(X_train, label=y_train,
                            categorical_feature=cat_cols, free_raw_data=False)
    lgb_valid = lgb.Dataset(X_valid, label=valid_df["demand"],
                            categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS,
        lgb_train,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        num_boost_round=NUM_BOOST_ROUND,
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
            lgb.log_evaluation(period=VERBOSE_EVAL)
        ]
    )

    valid_df = valid_df.copy()
    valid_df["pred"] = model.predict(X_valid, num_iteration=model.best_iteration)
    rmse = root_mean_squared_error(valid_df["demand"], valid_df["pred"])
    mae  = mean_absolute_error(valid_df["demand"], valid_df["pred"])

    row = {
        "exp":            exp_name,
        "added_features": ", ".join(added) if added else "None",
        "n_features":     len(feature_cols),
        "n_rows":         model_df.shape[0],
        "rmse":           rmse,
        "mae":            mae,
        "best_iteration": model.best_iteration
    }
    results.append(row)

    # Save per-experiment outputs
    pd.DataFrame([row]).to_csv(
        OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_metrics.csv", index=False)
    valid_df[["id", "date", "demand", "pred"]].to_csv(
        OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_valid_predictions.csv", index=False)
    pd.DataFrame({
        "feature":          feature_cols,
        "importance_gain":  model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split")
    }).sort_values("importance_gain", ascending=False).to_csv(
        OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_feature_importance.csv", index=False)

    print(f"RMSE: {rmse:.6f} | MAE: {mae:.6f} | best_iter: {model.best_iteration}")
    del model_df, train_df, valid_df, X_train, X_valid, lgb_train, lgb_valid, model
    gc.collect()

print("\n" + "=" * 70)
print("ALL EXPERIMENTS DONE")
results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(results_df)

Running: E0
Baseline only
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.98521	valid's rmse: 2.16199
Early stopping, best iteration is:
[97]	train's rmse: 1.98788	valid's rmse: 2.16194
RMSE: 2.161936 | MAE: 1.016549 | best_iter: 97
Running: E1
Added features: ['lag_56']
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.98678	valid's rmse: 2.15926
[200]	train's rmse: 1.93911	valid's rmse: 2.17044
Early stopping, best iteration is:
[107]	train's rmse: 1.98136	valid's rmse: 2.15775
RMSE: 2.157753 | MAE: 1.016246 | best_iter: 107
Running: E2
Added features: ['lag_365']
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.89882	valid's rmse: 2.19069
Early stopping, best iteration is:
[80]	train's rmse: 1.93543	valid's rmse: 2.18643
RMSE: 2.186433 | MAE: 1.024523 | best_iter: 80
Running: E3
Added features: ['rstd_28', 'rmean_56']
Training until validation scores don't improve for 100 rounds


,exp,added_features,n_features,n_rows,rmse,mae,best_iteration
0,E7_E1,"snap_flag, is_event, lag_56",17,944000,2.132212,1.011076,108
1,E9_E1,"price_vs_mean, price_vs_dept_avg, is_price_dro...",23,944000,2.135398,1.013002,112
2,E9,"price_vs_mean, price_vs_dept_avg, is_price_dro...",22,946000,2.136403,1.012673,112
3,E7_E5,"snap_flag, is_event, price_vs_mean, price_vs_d...",18,946000,2.140073,1.011736,123
4,E7,"snap_flag, is_event",16,946000,2.142124,1.012065,122
5,E1,lag_56,15,944000,2.157753,1.016246,107
6,E5,"price_vs_mean, price_vs_dept_avg",16,946000,2.158988,1.017276,99
7,E10,"lag_56, rmax_28, rmin_28",17,944000,2.159527,1.016193,115
8,E4,"rmax_28, rmin_28",16,946000,2.159556,1.016820,93
9,E8,"dept_avg_demand, store_avg_demand",16,946000,2.159977,1.017729,101


In [11]:
# =========================
# 11) Save combined summary
# =========================
summary_path = OUTPUT_DIR / f"{PIPELINE_NAME}_all_experiments_summary.csv"
results_df.to_csv(summary_path, index=False)
print("Summary saved to:", summary_path)
print()
print("=== BEST EXPERIMENT ===")
print(results_df.iloc[0])

Summary saved to: /content/drive/MyDrive/Group Project - Predictive/data/FE_output/combined_internal_features_all_experiments_summary.csv

=== BEST EXPERIMENT ===
exp                                     E7_E1
added_features    snap_flag, is_event, lag_56
n_features                                 17
n_rows                                 944000
rmse                                 2.132212
mae                                  1.011076
best_iteration                            108
Name: 0, dtype: object


In [12]:
# =========================
# External features reference (tested separately with cleaned_data_external)
# =========================
external_results = pd.DataFrame([
    {"exp": "ext_baseline",   "added_features": "None",                                          "rmse": 2.525101, "note": "baseline on external dataset"},
    {"exp": "ext_weather",    "added_features": "is_hot_day, is_cold_night, tmax_z, tmin_z",     "rmse": 2.1618,   "note": "weather only"},
    {"exp": "ext_snap_event", "added_features": "snap_flag, is_event",                           "rmse": 2.1379,   "note": "best external"},
    {"exp": "ext_all",        "added_features": "snap_flag, is_event + weather",                 "rmse": 2.1418,   "note": "all external"},
])
print("=== External Features Reference ===")
display(external_results)

=== External Features Reference ===


,exp,added_features,rmse,note
0,ext_baseline,None,2.525101,baseline on external dataset
1,ext_weather,"is_hot_day, is_cold_night, tmax_z, tmin_z",2.161800,weather only
2,ext_snap_event,"snap_flag, is_event",2.137900,best external
3,ext_all,"snap_flag, is_event + weather",2.141800,all external


In [14]:
# =========================
# 12) Feature Engineering Table + Export PDF
# =========================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

FEATURE_DIR = Path("../results")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# --- Feature Engineering Table ---
fe_table = pd.DataFrame([
    # Final selected features
    {"Feature Name": "lag_56",            "Category": "Internal", "Source": "demand",     "Description": "Sales 56 days ago",                          "Leakage-safe": "shift(56)",           "RMSE (alone)": 2.1578, "Keep": "✓"},
    {"Feature Name": "snap_flag",         "Category": "External", "Source": "calendar",   "Description": "SNAP benefit active in state on that day",    "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1421, "Keep": "✓"},
    {"Feature Name": "is_event",          "Category": "External", "Source": "calendar",   "Description": "Any event/holiday on that day",               "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1421, "Keep": "✓"},
    # Tested but not selected
    {"Feature Name": "lag_365",           "Category": "Internal", "Source": "demand",     "Description": "Sales 365 days ago",                          "Leakage-safe": "shift(365)",          "RMSE (alone)": 2.1864, "Keep": "✗ — too many NaNs"},
    {"Feature Name": "rstd_28",           "Category": "Internal", "Source": "demand",     "Description": "28-day rolling std of lag_28",                "Leakage-safe": "rolling on lag_28",   "RMSE (alone)": 2.1623, "Keep": "✗"},
    {"Feature Name": "rmax_28",           "Category": "Internal", "Source": "demand",     "Description": "28-day rolling max of lag_28",                "Leakage-safe": "rolling on lag_28",   "RMSE (alone)": 2.1596, "Keep": "✗"},
    {"Feature Name": "rmin_28",           "Category": "Internal", "Source": "demand",     "Description": "28-day rolling min of lag_28",                "Leakage-safe": "rolling on lag_28",   "RMSE (alone)": 2.1596, "Keep": "✗"},
    {"Feature Name": "price_vs_mean",     "Category": "Internal", "Source": "sell_price", "Description": "Price / item historical mean price",          "Leakage-safe": "historical mean",     "RMSE (alone)": 2.1590, "Keep": "✗"},
    {"Feature Name": "price_vs_dept_avg", "Category": "Internal", "Source": "sell_price", "Description": "Price / dept average price on same day",      "Leakage-safe": "same-day groupby",    "RMSE (alone)": 2.1590, "Keep": "✗"},
    {"Feature Name": "is_price_drop",     "Category": "Internal", "Source": "sell_price", "Description": "Price lower than previous day",               "Leakage-safe": "price_lag_1",         "RMSE (alone)": 2.1605, "Keep": "✗"},
    {"Feature Name": "is_price_min_7d",   "Category": "Internal", "Source": "sell_price", "Description": "Price is 7-day local minimum",                "Leakage-safe": "shift(1).rolling(7)", "RMSE (alone)": 2.1605, "Keep": "✗"},
    {"Feature Name": "dept_avg_demand",   "Category": "Internal", "Source": "demand",     "Description": "Dept-store avg demand, lag28 + rolling28",    "Leakage-safe": "shift(28).rolling(28)","RMSE (alone)": 2.1600, "Keep": "✗"},
    {"Feature Name": "store_avg_demand",  "Category": "Internal", "Source": "demand",     "Description": "Store avg demand, lag28 + rolling28",         "Leakage-safe": "shift(28).rolling(28)","RMSE (alone)": 2.1600, "Keep": "✗"},
    {"Feature Name": "is_hot_day",        "Category": "External", "Source": "weather.csv","Description": "Extreme high temperature day",                "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1618, "Keep": "✗"},
    {"Feature Name": "is_cold_night",     "Category": "External", "Source": "weather.csv","Description": "Extreme low temperature night",               "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1618, "Keep": "✗"},
    {"Feature Name": "tmax_z",            "Category": "External", "Source": "weather.csv","Description": "Max temperature z-score (state × month)",     "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1618, "Keep": "✗"},
    {"Feature Name": "tmin_z",            "Category": "External", "Source": "weather.csv","Description": "Min temperature z-score (state × month)",     "Leakage-safe": "known at prediction", "RMSE (alone)": 2.1618, "Keep": "✗"},
])

display(fe_table)

# --- Experiment Results Summary ---
experiment_summary = pd.DataFrame([
    # Internal
    {"Experiment": "E0",    "Feature Group": "Baseline",          "Added Features": "None",                           "RMSE": 2.1619, "vs Baseline": "—"},
    {"Experiment": "E1",    "Feature Group": "Demand lag",         "Added Features": "lag_56",                         "RMSE": 2.1578, "vs Baseline": "-0.0041"},
    {"Experiment": "E2",    "Feature Group": "Demand lag",         "Added Features": "lag_365",                        "RMSE": 2.1864, "vs Baseline": "+0.0245"},
    {"Experiment": "E3",    "Feature Group": "Rolling",            "Added Features": "rstd_28, rmean_56",              "RMSE": 2.1623, "vs Baseline": "+0.0004"},
    {"Experiment": "E4",    "Feature Group": "Rolling extreme",    "Added Features": "rmax_28, rmin_28",               "RMSE": 2.1596, "vs Baseline": "-0.0023"},
    {"Experiment": "E10",   "Feature Group": "Demand combo",       "Added Features": "lag_56, rmax_28, rmin_28",       "RMSE": 2.1595, "vs Baseline": "-0.0024"},
    {"Experiment": "E5",    "Feature Group": "Price",              "Added Features": "price_vs_mean, price_vs_dept_avg","RMSE": 2.1590, "vs Baseline": "-0.0029"},
    {"Experiment": "E6",    "Feature Group": "Promo",              "Added Features": "is_price_drop, is_price_min_7d", "RMSE": 2.1605, "vs Baseline": "-0.0014"},
    {"Experiment": "E7",    "Feature Group": "Event & SNAP",       "Added Features": "snap_flag, is_event",            "RMSE": 2.1421, "vs Baseline": "-0.0198"},
    {"Experiment": "E8",    "Feature Group": "Hierarchy",          "Added Features": "dept_avg_demand, store_avg_demand","RMSE": 2.1600, "vs Baseline": "-0.0019"},
    {"Experiment": "E9",    "Feature Group": "E5+E6+E7+E8",       "Added Features": "All E5~E8",                      "RMSE": 2.1364, "vs Baseline": "-0.0255"},
    # External
    {"Experiment": "ext_weather",    "Feature Group": "External — Weather", "Added Features": "is_hot_day, is_cold_night, tmax_z, tmin_z", "RMSE": 2.1618, "vs Baseline": "-0.0001"},
    {"Experiment": "ext_snap_event", "Feature Group": "External — Event",   "Added Features": "snap_flag, is_event",                       "RMSE": 2.1379, "vs Baseline": "-0.0240"},
    {"Experiment": "ext_all",        "Feature Group": "External — All",     "Added Features": "snap_flag, is_event + weather",             "RMSE": 2.1418, "vs Baseline": "-0.0201"},
    # Combinations
    {"Experiment": "E7_E1",  "Feature Group": "Combination ★ BEST", "Added Features": "snap_flag, is_event, lag_56",        "RMSE": 2.1322, "vs Baseline": "-0.0297"},
    {"Experiment": "E7_E5",  "Feature Group": "Combination",        "Added Features": "snap_flag, is_event, price features", "RMSE": 2.1401, "vs Baseline": "-0.0218"},
    {"Experiment": "E9_E1",  "Feature Group": "Combination",        "Added Features": "All E5~E8 + lag_56",                 "RMSE": 2.1354, "vs Baseline": "-0.0265"},
])

print("\n=== Experiment Results Summary ===")
display(experiment_summary)

# --- Export CSV ---
fe_table_path = FEATURE_DIR / "feature_engineering_table.csv"
experiment_summary_path = FEATURE_DIR / "experiment_results_summary.csv"

fe_table.to_csv(fe_table_path, index=False)
experiment_summary.to_csv(experiment_summary_path, index=False)

print(f"Saved: {fe_table_path}")
print(f"Saved: {experiment_summary_path}")


,Feature Name,Category,Source,Description,Leakage-safe,RMSE (alone),Keep
0,lag_56,Internal,demand,Sales 56 days ago,shift(56),2.1578,✓
1,snap_flag,External,calendar,SNAP benefit active in state on that day,known at prediction,2.1421,✓
2,is_event,External,calendar,Any event/holiday on that day,known at prediction,2.1421,✓
3,lag_365,Internal,demand,Sales 365 days ago,shift(365),2.1864,✗ — too many NaNs
4,rstd_28,Internal,demand,28-day rolling std of lag_28,rolling on lag_28,2.1623,✗
5,rmax_28,Internal,demand,28-day rolling max of lag_28,rolling on lag_28,2.1596,✗
6,rmin_28,Internal,demand,28-day rolling min of lag_28,rolling on lag_28,2.1596,✗
7,price_vs_mean,Internal,sell_price,Price / item historical mean price,historical mean,2.1590,✗
8,price_vs_dept_avg,Internal,sell_price,Price / dept average price on same day,same-day groupby,2.1590,✗
9,is_price_drop,Internal,sell_price,Price lower than previous day,price_lag_1,2.1605,✗



=== Experiment Results Summary ===


,Experiment,Feature Group,Added Features,RMSE,vs Baseline
0,E0,Baseline,None,2.1619,—
1,E1,Demand lag,lag_56,2.1578,-0.0041
2,E2,Demand lag,lag_365,2.1864,+0.0245
3,E3,Rolling,"rstd_28, rmean_56",2.1623,+0.0004
4,E4,Rolling extreme,"rmax_28, rmin_28",2.1596,-0.0023
5,E10,Demand combo,"lag_56, rmax_28, rmin_28",2.1595,-0.0024
6,E5,Price,"price_vs_mean, price_vs_dept_avg",2.1590,-0.0029
7,E6,Promo,"is_price_drop, is_price_min_7d",2.1605,-0.0014
8,E7,Event & SNAP,"snap_flag, is_event",2.1421,-0.0198
9,E8,Hierarchy,"dept_avg_demand, store_avg_demand",2.1600,-0.0019


Saved: /content/drive/MyDrive/Group Project - Predictive/Feature/feature_engineering_table.csv
Saved: /content/drive/MyDrive/Group Project - Predictive/Feature/experiment_results_summary.csv


In [17]:
# =========================
# Step 6 — Output final engineered feature file (FAST_MODE=False, full data)
# =========================
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

DATA_DIR    = Path("../cleaned_data")  # output of notebook 01
FEATURE_DIR = Path("../results")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Load full data — no sampling
needed_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "event_name_1", "snap_flag", "is_active"
]
available_cols = pq.read_schema(DATA_DIR / "long_df_clean.parquet").names
needed_cols = [c for c in needed_cols if c in available_cols]

df_full = pd.read_parquet(DATA_DIR / "long_df_clean.parquet", columns=needed_cols)
df_full["date"] = pd.to_datetime(df_full["date"])
df_full = df_full.sort_values(["id", "date"]).reset_index(drop=True)
print("Full data loaded:", df_full.shape)

# --- Baseline features ---
df_full["dow"]   = df_full["date"].dt.dayofweek.astype("int8")
df_full["month"] = df_full["date"].dt.month.astype("int8")

price_grp = df_full.groupby("id", sort=False)["sell_price"]
df_full["price_lag_1"]  = price_grp.shift(1)
df_full["price_change"] = (
    (df_full["sell_price"] - df_full["price_lag_1"]) / df_full["price_lag_1"]
).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
df_full["is_promo"] = (df_full["sell_price"] < df_full["price_lag_1"].fillna(df_full["sell_price"])).astype("int8")

demand_grp = df_full.groupby("id", sort=False)["demand"]
df_full["lag_7"]    = demand_grp.shift(7)
df_full["lag_28"]   = demand_grp.shift(28)
df_full["rmean_7"]  = df_full.groupby("id", sort=False)["lag_7"].transform(lambda x: x.rolling(7).mean())
df_full["rmean_28"] = df_full.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).mean())

for col in ["sell_price", "price_lag_1", "lag_7", "lag_28", "rmean_7", "rmean_28"]:
    df_full[col] = pd.to_numeric(df_full[col], downcast="float")

# --- E7_E1 features ---
# lag_56
df_full["lag_56"] = demand_grp.shift(56)
df_full["lag_56"] = pd.to_numeric(df_full["lag_56"], downcast="float")

# snap_flag (already in parquet)
df_full["snap_flag"] = df_full["snap_flag"].astype("int8")

# is_event
df_full["event_name_1"] = df_full["event_name_1"].fillna("NoEvent")
df_full["is_event"] = (df_full["event_name_1"] != "NoEvent").astype("int8")

gc.collect()

# --- Select final columns ---
final_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "is_active",
    # baseline features
    "sell_price", "price_change", "is_promo",
    "dow", "month",
    "lag_7", "lag_28", "rmean_7", "rmean_28",
    # E7_E1 features (best combination)
    "lag_56", "snap_flag", "is_event"
]

df_output = df_full[final_cols].copy()
del df_full; gc.collect()

print("Output shape:", df_output.shape)
print("Columns:", df_output.columns.tolist())
print("NaN counts:")
print(df_output.isnull().sum()[df_output.isnull().sum() > 0])

# --- Save ---
output_path = FEATURE_DIR / "long_df_with_features.parquet"  # loaded by notebooks 04a, 04c, 05
df_output.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")


Full data loaded: (16098720, 15)
Output shape: (16098720, 21)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'date', 'demand', 'is_active', 'sell_price', 'price_change', 'is_promo', 'dow', 'month', 'lag_7', 'lag_28', 'rmean_7', 'rmean_28', 'lag_56', 'snap_flag', 'is_event']
NaN counts:
lag_7        213430
lag_28       853720
rmean_7      396370
rmean_28    1676950
lag_56      1707440
dtype: int64
Saved to: /content/drive/MyDrive/Group Project - Predictive/Feature/long_df_with_features.parquet
File size: 53.8 MB
